Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 02 · Prices, returns and capitalization

**Question:** How do we construct comparable histories without inventing observations?

Frozen October 12 prices, split-adjusted closes, volume, ETF histories, supplementary security histories and FMP capitalization feed the shared loader. Study estimates stop September 30 even though descriptive processed files span the full vintage.

Research report · Method and code guide · Evidence map

**Collection record:** original execution outputs are retained. Read this notebook for provenance; use the offline reproduction command to rebuild the research without refreshing downloads.


**Chapter route**

Measurement and interpretation principles · Data and design context · Read and run this notebook · Calculation source: foundation.loader · Calculation source: foundation.build · Results, examples and interpretation · Collection boundary · descriptive portfolio · capitalization and issuer identity · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Daily simple returns are $r_t=P_t/P_{t-1}-1$ only across consecutive valid sessions. A missing quote must not turn a multi-day change into a one-day observation. Compatible split-adjusted close and share volume give dollar turnover; issuer capitalization is taken once per issuer, rather than summed across duplicate share classes.


## Data and design context

The guarded collection cells document the frozen October 12 price vintage. The cleaning definitions below handle supplementary histories, same-security windows, zero-volume dates and accepted event corrections. Processed descriptive tables cover the full vintage; study estimates stop September 30, 2026.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version. The original collection cells below remain archival; do not rerun them merely to read the calculations.


## Calculation source: foundation.loader

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Shared study loader: the single implementation of the price cleaning first written in notebooks/02_prices.ipynb.

Every later analysis reads prices, returns, factors and market cap through these functions, never through the raw
Yahoo or FMP files. Each input file is checked against the sha256 in its manifest before it is used.

Cleaning, in order (identical to notebook 02, which now calls this module)
1. Read the frozen 2026-10-12 Yahoo vintage by explicit name (adjusted close, split-adjusted close, volume). The
   2026-10-13 leftovers are never read.
2. Replace the columns of recovered tickers (WBD from FMP, PSKY from Yahoo under SKYD) with the supplement file.
3. Apply the same-security windows (same_security.py): sessions before history_start or after history_end are missing.
4. Zero-volume rule: a session with no trades is missing (inside the same-security window; outside it the window is
   the reason).
5. Corrections (apply_corrections=True): the extreme-event ledger, if present, is applied to the cleaned adjusted
   close (corrections.py). Returns are always recomputed from the corrected panel.
6. Daily simple total return r_t = P_t / P_t-1 - 1 on the adjusted close; missing whenever either price is missing,
   so missing sessions are never bridged.
7. Truncate to through= (default settings.research_price_cutoff, 2026-09-30). Corrections are applied before the
   truncation so a back adjustment is the same whatever the cutoff.

Every loader function takes through= and apply_corrections=; functions whose output does not depend on one of them
accept it so every call in the study reads the same way.
"""

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation.corrections import apply_extreme_event_ledger, empty_extreme_event_ledger, read_extreme_event_ledger
from quiet_index_500.foundation.manifests import verify_file_against_manifest
from quiet_index_500.foundation import same_security

# folders, read at call time so tests can point them at synthetic files
raw_data_folder = settings.raw_data_folder
processed_data_folder = settings.processed_data_folder
expected_universe_size = 503  # security records in the 30 Sep 2026 IVV snapshot

factor_columns = ["mkt_rf", "smb", "hml", "rmw", "cma", "rf", "mom"]
gics_columns = ["gics_sector_code", "gics_sector", "gics_industry_group_code", "gics_industry_group",
                "gics_industry_code", "gics_industry", "gics_sub_industry_code", "gics_sub_industry"]


# ---------- file locations (the vintage is named explicitly, never "newest matching file") ----------


**universe_tickers_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [2]:
def universe_tickers_path():
    return processed_data_folder / f"universe_tickers_{settings.holdings_date}.csv"


**universe_weights_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [3]:
def universe_weights_path():
    return processed_data_folder / f"universe_weights_{settings.holdings_date}.csv"


**ivv_equity_holdings_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [4]:
def ivv_equity_holdings_path():
    return processed_data_folder / f"ivv_equity_holdings_{settings.holdings_date}.csv"


**adjusted_close_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [5]:
def adjusted_close_path():
    return raw_data_folder / f"prices_adjusted_close_through_{settings.price_vintage_last_session}.csv"


**split_adjusted_close_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [6]:
def split_adjusted_close_path():
    return raw_data_folder / f"prices_close_split_adjusted_through_{settings.price_vintage_last_session}.csv"


**volume_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [7]:
def volume_path():
    return raw_data_folder / f"prices_volume_through_{settings.price_vintage_last_session}.csv"


**price_pull_manifest_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [8]:
def price_pull_manifest_path():
    return raw_data_folder / f"prices_through_{settings.price_vintage_last_session}.manifest.json"


**supplement_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [9]:
def supplement_path():
    return raw_data_folder / f"prices_supplement_through_{settings.price_vintage_last_session}.csv"


**spy_prices_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [10]:
def spy_prices_path():
    return raw_data_folder / f"prices_spy_through_{settings.price_vintage_last_session}.csv"


**smh_prices_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [11]:
def smh_prices_path():
    return raw_data_folder / f"prices_smh_through_{settings.price_vintage_last_session}.csv"


**market_cap_raw_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [12]:
def market_cap_raw_path():
    return raw_data_folder / f"market_cap_fmp_through_{settings.price_vintage_last_session}.csv"


**factors_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [13]:
def factors_path():
    return processed_data_folder / settings.ff5_mom_factors_file


# ---------- helpers ----------


**_through_timestamp**

The cutoff as a Timestamp; a cutoff after the frozen vintage is an error, not a silent truncation.


In [14]:
def _through_timestamp(through):
    """The cutoff as a Timestamp; a cutoff after the frozen vintage is an error, not a silent truncation."""
    through_timestamp = pd.Timestamp(through)
    if through_timestamp > pd.Timestamp(settings.price_vintage_last_session):
        raise ValueError(f"through={through} is after the price vintage's last session {settings.price_vintage_last_session}")
    return through_timestamp


**_read_verified_csv**

Read a CSV once per session after checking its fingerprint; callers receive copies.


In [15]:
@lru_cache(maxsize=None)
def _read_verified_csv(path_text, manifest_path_text, index_column):
    """Read a CSV once per session after checking its fingerprint; callers receive copies."""
    verify_file_against_manifest(Path(path_text), Path(manifest_path_text) if manifest_path_text else None)
    if index_column is None:
        return pd.read_csv(path_text)
    return pd.read_csv(path_text, index_col=index_column, parse_dates=index_column == "date")


**_verified_csv**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [16]:
def _verified_csv(file_path, index_column=None, manifest_path=None):
    return _read_verified_csv(str(file_path), str(manifest_path) if manifest_path else None, index_column).copy()


**clear_cache**

Forget every file read in this session (after a file legitimately changes, or between tests).


In [17]:
def clear_cache():
    """Forget every file read in this session (after a file legitimately changes, or between tests)."""
    _read_verified_csv.cache_clear()
    _cleaned_full_vintage.cache_clear()


# ---------- universe, weights, identifiers ----------


**load_universe_tickers**

The 503 snapshot tickers in the frozen order, as a tuple (fingerprint-checked against notebook 01's manifest).


In [18]:
def load_universe_tickers(through=settings.research_price_cutoff, apply_corrections=True):
    """The 503 snapshot tickers in the frozen order, as a tuple (fingerprint-checked against notebook 01's manifest)."""
    universe_tickers = tuple(_verified_csv(universe_tickers_path())["ticker"])  # a tuple cannot be edited, so the list stays constant
    assert len(universe_tickers) == expected_universe_size and len(set(universe_tickers)) == expected_universe_size
    return universe_tickers


**load_snapshot_weights**

Precise snapshot weights (decimal, sum to 1) indexed by ticker, in universe order.


In [19]:
def load_snapshot_weights(through=settings.research_price_cutoff, apply_corrections=True):
    """Precise snapshot weights (decimal, sum to 1) indexed by ticker, in universe order."""
    universe_weights = _verified_csv(universe_weights_path(), index_column="ticker")
    return universe_weights["weight"].reindex(list(load_universe_tickers()))  # same order as the price columns


**representative_security_by_ticker**

For each ticker, its issuer's representative security: the class with the largest snapshot weight.


In [20]:
def representative_security_by_ticker(cik_by_ticker, snapshot_weights):
    """For each ticker, its issuer's representative security: the class with the largest snapshot weight."""
    representative_by_cik = snapshot_weights.groupby(cik_by_ticker).idxmax()  # one ticker per CIK
    return cik_by_ticker.map(representative_by_cik)


**load_identifiers**

One row per snapshot security: cik, issuer, snapshot weight, representative-security flag and the four GICS levels.

GICS is the 30 Sep 2026 classification applied retrospectively. The representative security is fixed for the study
(largest snapshot weight within the issuer: GOOGL, FOXA, NWSA; every other issuer has one class).


In [21]:
def load_identifiers(through=settings.research_price_cutoff, apply_corrections=True):
    """One row per snapshot security: cik, issuer, snapshot weight, representative-security flag and the four GICS levels.

    GICS is the 30 Sep 2026 classification applied retrospectively. The representative security is fixed for the study
    (largest snapshot weight within the issuer: GOOGL, FOXA, NWSA; every other issuer has one class).
    """
    universe_tickers = list(load_universe_tickers())
    ivv_equity_holdings = _verified_csv(ivv_equity_holdings_path(), index_column="ticker").reindex(universe_tickers)
    snapshot_weights = load_snapshot_weights()
    cik_by_ticker = ivv_equity_holdings["cik"].astype("int64")
    representative_security = representative_security_by_ticker(cik_by_ticker, snapshot_weights)
    identifiers = pd.DataFrame({
        "cik": cik_by_ticker,
        "issuer_name": ivv_equity_holdings["issuer_name"],
        "ivv_name": ivv_equity_holdings["Name"],
        "snapshot_weight": snapshot_weights,
        "issuer_snapshot_weight": snapshot_weights.groupby(cik_by_ticker).transform("sum"),  # both classes together, for company-level weighting
        "share_classes_of_issuer": cik_by_ticker.map(cik_by_ticker.value_counts()),
        "representative_security": representative_security,
        "is_representative_security": representative_security == cik_by_ticker.index,
    }).rename_axis("ticker")
    return identifiers.join(ivv_equity_holdings[gics_columns])


# ---------- prices ----------


**PricePanels**

Cleaned price panels (date index, one column per ticker) and the masks that explain every missing value.


In [22]:
@dataclass
class PricePanels:
    """Cleaned price panels (date index, one column per ticker) and the masks that explain every missing value."""
    adjusted_close: pd.DataFrame          # splits and dividends; total-return basis; corrections applied when switched on
    split_adjusted_close: pd.DataFrame    # splits only; the price for market cap and dollar volume
    volume: pd.DataFrame                  # shares traded
    zero_volume_days: pd.DataFrame        # True where the zero-volume rule removed the session
    before_same_security_start: pd.DataFrame  # True before a security's same-security history starts
    after_history_end: pd.DataFrame       # True after a recorded history end (WBD after 2026-09-30)
    quarantined_returns: pd.DataFrame     # True where a ledger quarantine made the return missing
    quarantined_prices: pd.DataFrame      # True where a ledger quarantine made the adjusted close missing
    applied_corrections: pd.DataFrame     # one row per ledger action taken


**with_supplement**

The Yahoo panel with each recovered ticker's column replaced by its supplementary series (notebook 02's rule).


In [23]:
def with_supplement(vendor_panel, supplement, field_name):
    """The Yahoo panel with each recovered ticker's column replaced by its supplementary series (notebook 02's rule)."""
    combined_panel = vendor_panel.copy()
    for ticker, rows in supplement.groupby("ticker"):
        combined_panel[ticker] = rows.set_index("date")[field_name].reindex(vendor_panel.index)
    return combined_panel


**clean_price_panels**

Supplement, same-security windows and the zero-volume rule, applied to the downloaded panels.

Returns a dict with adjusted_close, split_adjusted_close, volume, zero_volume_days, before_same_security_start and
after_history_end, every one aligned to the downloaded sessions and tickers.


In [24]:
def clean_price_panels(adjusted_close_raw, split_adjusted_close_raw, volume_raw, supplement, rules):
    """Supplement, same-security windows and the zero-volume rule, applied to the downloaded panels.

    Returns a dict with adjusted_close, split_adjusted_close, volume, zero_volume_days, before_same_security_start and
    after_history_end, every one aligned to the downloaded sessions and tickers.
    """
    assert adjusted_close_raw.index.is_unique and adjusted_close_raw.index.is_monotonic_increasing  # one row per session, in date order
    assert adjusted_close_raw.index.equals(volume_raw.index) and adjusted_close_raw.index.equals(split_adjusted_close_raw.index)
    assert supplement["date"].isin(adjusted_close_raw.index).all()  # every supplementary date is a session of the main pull

    adjusted_close_vendor = with_supplement(adjusted_close_raw, supplement, "adjusted_close")
    split_adjusted_close_vendor = with_supplement(split_adjusted_close_raw, supplement, "close_split_adjusted")
    volume_vendor = with_supplement(volume_raw, supplement, "volume")

    rules_in_panel = rules[rules.index.isin(adjusted_close_vendor.columns)]  # a rule for a ticker outside the panel has nothing to act on
    before_same_security_start = pd.DataFrame(False, index=adjusted_close_vendor.index, columns=adjusted_close_vendor.columns)
    after_history_end = before_same_security_start.copy()
    for ticker, rule in rules_in_panel.iterrows():
        if pd.notna(rule["history_start"]):
            before_same_security_start[ticker] = adjusted_close_vendor.index < pd.Timestamp(rule["history_start"])
        if pd.notna(rule["history_end"]):
            after_history_end[ticker] = adjusted_close_vendor.index > pd.Timestamp(rule["history_end"])
    outside_same_security_history = before_same_security_start | after_history_end

    zero_volume_days = (volume_vendor == 0) & ~outside_same_security_history  # days outside the history are excluded for that reason instead
    return {
        "adjusted_close": adjusted_close_vendor.mask(zero_volume_days | outside_same_security_history),
        "split_adjusted_close": split_adjusted_close_vendor.mask(zero_volume_days | outside_same_security_history),
        "volume": volume_vendor.mask(zero_volume_days | outside_same_security_history),
        "zero_volume_days": zero_volume_days,
        "before_same_security_start": before_same_security_start,
        "after_history_end": after_history_end,
    }


**_cleaned_full_vintage**

The cleaned panels over the whole 2026-10-12 vintage, before corrections and truncation (computed once).


In [25]:
@lru_cache(maxsize=None)
def _cleaned_full_vintage():
    """The cleaned panels over the whole 2026-10-12 vintage, before corrections and truncation (computed once)."""
    universe_tickers = load_universe_tickers()
    adjusted_close_raw = _verified_csv(adjusted_close_path(), index_column="date", manifest_path=price_pull_manifest_path())
    split_adjusted_close_raw = _verified_csv(split_adjusted_close_path(), index_column="date", manifest_path=price_pull_manifest_path())
    volume_raw = _verified_csv(volume_path(), index_column="date", manifest_path=price_pull_manifest_path())
    assert tuple(adjusted_close_raw.columns) == universe_tickers  # same tickers, same order as the frozen universe
    supplement = _verified_csv(supplement_path())
    supplement["date"] = pd.to_datetime(supplement["date"])
    return clean_price_panels(adjusted_close_raw, split_adjusted_close_raw, volume_raw, supplement, same_security.same_security_rules())


**load_price_panels**

Cleaned adjusted close, split-adjusted close and volume with their masks, through the cutoff (PricePanels).


In [26]:
def load_price_panels(through=settings.research_price_cutoff, apply_corrections=True):
    """Cleaned adjusted close, split-adjusted close and volume with their masks, through the cutoff (PricePanels)."""
    through_timestamp = _through_timestamp(through)
    cleaned_panels = {name: panel.copy() for name, panel in _cleaned_full_vintage().items()}
    ledger = read_extreme_event_ledger() if apply_corrections else empty_extreme_event_ledger()
    corrected = apply_extreme_event_ledger(cleaned_panels["adjusted_close"], ledger)  # on the full vintage, then truncated
    within_cutoff = cleaned_panels["adjusted_close"].index <= through_timestamp
    return PricePanels(
        adjusted_close=corrected.adjusted_close[within_cutoff],
        split_adjusted_close=cleaned_panels["split_adjusted_close"][within_cutoff],
        volume=cleaned_panels["volume"][within_cutoff],
        zero_volume_days=cleaned_panels["zero_volume_days"][within_cutoff],
        before_same_security_start=cleaned_panels["before_same_security_start"][within_cutoff],
        after_history_end=cleaned_panels["after_history_end"][within_cutoff],
        quarantined_returns=corrected.quarantined_returns[within_cutoff],
        quarantined_prices=corrected.quarantined_prices[within_cutoff],
        applied_corrections=corrected.applied_corrections,
    )


**daily_simple_returns_from_prices**

r_t = P_t / P_t-1 - 1 between consecutive sessions; missing on the first row, after any missing price and where quarantined.


In [27]:
def daily_simple_returns_from_prices(adjusted_close, quarantined_returns=None):
    """r_t = P_t / P_t-1 - 1 between consecutive sessions; missing on the first row, after any missing price and where quarantined."""
    daily_simple_returns = adjusted_close / adjusted_close.shift(1) - 1  # a missing price on either day leaves the return missing: no bridging
    if quarantined_returns is not None and quarantined_returns.to_numpy().any():
        daily_simple_returns = daily_simple_returns.mask(quarantined_returns)
    assert daily_simple_returns.min().min() > -1  # no return at or below -100%, which would mean a zero or negative price
    return daily_simple_returns


**load_daily_returns**

Daily simple total returns of the 503 securities from the cleaned (and corrected) adjusted close.


In [28]:
def load_daily_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """Daily simple total returns of the 503 securities from the cleaned (and corrected) adjusted close."""
    price_panels = load_price_panels(through=through, apply_corrections=apply_corrections)
    return daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns)


**_etf_daily_returns**

Daily simple total return of one ETF from Yahoo's adjusted close, zero-volume sessions missing (notebook 02's rule).


In [29]:
def _etf_daily_returns(prices_path, return_name, through):
    """Daily simple total return of one ETF from Yahoo's adjusted close, zero-volume sessions missing (notebook 02's rule)."""
    etf_prices_raw = _verified_csv(prices_path, index_column="date")
    stock_sessions = _cleaned_full_vintage()["adjusted_close"].index
    assert etf_prices_raw.index.equals(stock_sessions)  # the ETF and the stocks share every session
    etf_adjusted_close = etf_prices_raw["adjusted_close"].mask(etf_prices_raw["volume"] == 0)
    etf_daily_returns = (etf_adjusted_close / etf_adjusted_close.shift(1) - 1).rename(return_name)
    return etf_daily_returns[etf_daily_returns.index <= _through_timestamp(through)]


**load_spy_returns**

SPY daily simple total return (missing on the first session); the benchmark for ordinary and downside beta.


In [30]:
def load_spy_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """SPY daily simple total return (missing on the first session); the benchmark for ordinary and downside beta."""
    return _etf_daily_returns(spy_prices_path(), "spy_return", through)


**load_smh_returns**

SMH daily simple total return (missing on the first session); minus rf it is the SMH factor.


In [31]:
def load_smh_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """SMH daily simple total return (missing on the first session); minus rf it is the SMH factor."""
    return _etf_daily_returns(smh_prices_path(), "smh_return", through)


# ---------- factors and excess returns ----------


**load_factors**

French FF5 plus mom, daily, decimal, CRSP vintage 202608, date-indexed, rows on or before the cutoff.


In [32]:
def load_factors(through=settings.research_price_cutoff, apply_corrections=True):
    """French FF5 plus mom, daily, decimal, CRSP vintage 202608, date-indexed, rows on or before the cutoff."""
    factors = _verified_csv(factors_path(), index_column="date")[factor_columns]
    return factors[factors.index <= _through_timestamp(through)]


**factor_data_last_date**

The factor cutoff read from the data: the last date of the factor file on or before through (2026-08-31).


In [33]:
def factor_data_last_date(through=settings.research_price_cutoff, apply_corrections=True):
    """The factor cutoff read from the data: the last date of the factor file on or before through (2026-08-31)."""
    return load_factors(through=through).index.max()


**excess_returns**

Returns minus rf on the same date, subtracted once; dates without an rf (after the factor cutoff) are missing.


In [34]:
def excess_returns(returns, risk_free_rate):
    """Returns minus rf on the same date, subtracted once; dates without an rf (after the factor cutoff) are missing."""
    risk_free_rate_aligned = risk_free_rate.reindex(returns.index)
    if isinstance(returns, pd.Series):
        return returns - risk_free_rate_aligned
    return returns.sub(risk_free_rate_aligned, axis=0)


**load_excess_returns**

Daily excess returns of the 503 securities (missing after the factor cutoff).


In [35]:
def load_excess_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """Daily excess returns of the 503 securities (missing after the factor cutoff)."""
    return excess_returns(load_daily_returns(through=through, apply_corrections=apply_corrections), load_factors(through=through)["rf"])


**load_spy_excess_returns**

SPY excess return: the market for ordinary and downside beta.


In [36]:
def load_spy_excess_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """SPY excess return: the market for ordinary and downside beta."""
    return excess_returns(load_spy_returns(through=through), load_factors(through=through)["rf"]).rename("spy_excess_return")


**load_smh_excess_returns**

SMH excess return: the SMH factor.


In [37]:
def load_smh_excess_returns(through=settings.research_price_cutoff, apply_corrections=True):
    """SMH excess return: the SMH factor."""
    return excess_returns(load_smh_returns(through=through), load_factors(through=through)["rf"]).rename("smh_excess_return")


# ---------- market cap (licensed FMP data) ----------


**clean_market_cap**

Same-security market cap by ticker (notebook 02's rule) and the masks behind it.

A value counts only on a day the security has a usable adjusted close, and never before market_cap_valid_from.
Returns (market_cap, market_cap_rejected, market_cap_missing_while_priced).


In [38]:
def clean_market_cap(market_cap_raw, adjusted_close, rules):
    """Same-security market cap by ticker (notebook 02's rule) and the masks behind it.

    A value counts only on a day the security has a usable adjusted close, and never before market_cap_valid_from.
    Returns (market_cap, market_cap_rejected, market_cap_missing_while_priced).
    """
    assert market_cap_raw.index.isin(adjusted_close.index).all()  # no market-cap date outside the price sessions
    market_cap_vendor = market_cap_raw.reindex(adjusted_close.index)
    market_cap_share_basis_rejected = pd.DataFrame(False, index=market_cap_vendor.index, columns=market_cap_vendor.columns)
    for ticker, rule in rules[rules.index.isin(market_cap_vendor.columns)].iterrows():
        if pd.notna(rule["market_cap_valid_from"]):
            market_cap_share_basis_rejected[ticker] = market_cap_vendor.index < pd.Timestamp(rule["market_cap_valid_from"])
    market_cap = market_cap_vendor.where(adjusted_close.notna()).mask(market_cap_share_basis_rejected)
    market_cap_rejected = market_cap_vendor.notna() & market_cap.isna()            # predecessor or wrong-basis values removed
    market_cap_missing_while_priced = adjusted_close.notna() & market_cap.isna()   # excludes a formation from size-dependent analysis, never the returns
    return market_cap, market_cap_rejected, market_cap_missing_while_priced


**issuer_market_cap_from_tickers**

One market cap per issuer (columns are CIKs, ascending): the class with the larger snapshot weight carries it.

FMP prices the issuer's full share count at each class's own price, so classes are never summed.


In [39]:
def issuer_market_cap_from_tickers(market_cap, cik_by_ticker, snapshot_weights):
    """One market cap per issuer (columns are CIKs, ascending): the class with the larger snapshot weight carries it.

    FMP prices the issuer's full share count at each class's own price, so classes are never summed.
    """
    issuer_market_cap_columns = {}
    for cik, tickers_of_issuer in cik_by_ticker.groupby(cik_by_ticker):
        class_tickers = list(tickers_of_issuer.index)
        issuer_figure_ticker = snapshot_weights[class_tickers].idxmax()
        issuer_market_cap_columns[cik] = market_cap[issuer_figure_ticker]
    issuer_market_cap = pd.DataFrame(issuer_market_cap_columns).rename_axis(columns="cik")
    assert issuer_market_cap.shape[1] == cik_by_ticker.nunique()
    return issuer_market_cap


**_market_cap_raw**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [40]:
def _market_cap_raw():
    market_cap_raw = _verified_csv(market_cap_raw_path(), index_column="date")
    assert tuple(market_cap_raw.columns) == load_universe_tickers()
    return market_cap_raw


**load_market_cap_same_security**

Ticker-level FMP market cap (USD) on same-security days with a usable price.

nonpositive_as_missing=True (the study default) sets zero or negative values missing for size measurement
(HONA, 2026-06-26); the return on that day is kept. The descriptive processed table keeps them (False).
A price quarantined by the ledger also removes that day's market cap, by the usable-price rule.


In [41]:
def load_market_cap_same_security(through=settings.research_price_cutoff, apply_corrections=True, nonpositive_as_missing=True):
    """Ticker-level FMP market cap (USD) on same-security days with a usable price.

    nonpositive_as_missing=True (the study default) sets zero or negative values missing for size measurement
    (HONA, 2026-06-26); the return on that day is kept. The descriptive processed table keeps them (False).
    A price quarantined by the ledger also removes that day's market cap, by the usable-price rule.
    """
    price_panels = load_price_panels(through=settings.price_vintage_last_session, apply_corrections=apply_corrections)
    market_cap, _, _ = clean_market_cap(_market_cap_raw(), price_panels.adjusted_close, same_security.same_security_rules())
    if nonpositive_as_missing:
        market_cap = market_cap.where(market_cap > 0)
    return market_cap[market_cap.index <= _through_timestamp(through)]


**load_issuer_market_cap**

One FMP market cap per issuer (columns are SEC CIKs), from the class with the larger snapshot weight.


In [42]:
def load_issuer_market_cap(through=settings.research_price_cutoff, apply_corrections=True, nonpositive_as_missing=True):
    """One FMP market cap per issuer (columns are SEC CIKs), from the class with the larger snapshot weight."""
    market_cap = load_market_cap_same_security(through=through, apply_corrections=apply_corrections, nonpositive_as_missing=nonpositive_as_missing)
    identifiers = load_identifiers()
    return issuer_market_cap_from_tickers(market_cap, identifiers["cik"], identifiers["snapshot_weight"])


## Calculation source: foundation.build

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [43]:
"""Regenerate every processed table notebooks/02_prices.ipynb writes, using the shared loader.

These tables are descriptive and cover the whole frozen vintage (through 2026-10-12), as they did before the loader
existed; the study loader itself defaults to the 2026-09-30 research cutoff. Each writer returns the table it wrote
and leaves a manifest beside it naming every input and the source code that built it.

Tables (in data/processed unless output_folder says otherwise)
- same_security_rules_2026-09-30.csv
- snapshot_portfolio_daily_returns_through_<vintage>.csv
- spy_daily_returns_through_<vintage>.csv, smh_daily_returns_through_<vintage>.csv
- portfolio_exclusions_through_<vintage>.csv
- ticker_data_flags_through_<vintage>.csv (extended with market_cap_nonpositive_days and quarantined_return_days)
- market_cap_same_security_through_<vintage>.csv, issuer_market_cap_through_<vintage>.csv (licensed: CSVs git-ignored)
"""

from pathlib import Path

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import corrections, loader, manifests, same_security
from quiet_index_500.foundation.manifests import write_manifest

default_built_by = "notebooks/02_prices.ipynb"
minimum_sessions_for_one_year = 252  # threshold for the under-one-year flag, carried from notebook 02
large_move_threshold = settings.extreme_return_threshold  # a daily simple return beyond +-50% is flagged for review, not removed
source_modules_used = [settings, loader, corrections, same_security, manifests]  # build itself is added below, once it exists


**_source_modules**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [44]:
def _source_modules():
    import quiet_index_500.foundation.build as build_module
    return source_modules_used + [build_module]


**_ledger_inputs**

The ledger file as an input, when corrections are on and the file exists.


In [45]:
def _ledger_inputs(apply_corrections):
    """The ledger file as an input, when corrections are on and the file exists."""
    ledger_path = corrections.extreme_event_ledger_path()
    return [ledger_path] if apply_corrections and ledger_path.exists() else []


**_vintage_table_path**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [46]:
def _vintage_table_path(output_folder, stem, through):
    return Path(output_folder) / f"{stem}_through_{pd.Timestamp(through).strftime('%Y-%m-%d')}.csv"


# ---------- pure calculations ----------


**snapshot_portfolio_daily_returns**

Daily return of the snapshot held at constant weights, renormalized each day over the names with a return.

Returns a date-indexed frame (first session dropped: no returns yet) with snapshot_portfolio_return,
weight_with_return (share of snapshot weight covered) and names_with_return.


In [47]:
def snapshot_portfolio_daily_returns(daily_simple_returns, snapshot_weights):
    """Daily return of the snapshot held at constant weights, renormalized each day over the names with a return.

    Returns a date-indexed frame (first session dropped: no returns yet) with snapshot_portfolio_return,
    weight_with_return (share of snapshot weight covered) and names_with_return.
    """
    has_return_today = daily_simple_returns.notna()
    weighted_return_sum = (daily_simple_returns.fillna(0) * snapshot_weights).sum(axis=1)  # a missing return adds nothing to the numerator
    weight_with_return = (has_return_today * snapshot_weights).sum(axis=1)                 # the share of snapshot weight that traded that day
    snapshot_portfolio_return = weighted_return_sum / weight_with_return                    # renormalize over the names with a return (Sai, 13 Oct 2026)
    portfolio_daily_returns = pd.DataFrame({
        "snapshot_portfolio_return": snapshot_portfolio_return,
        "weight_with_return": weight_with_return,
        "names_with_return": has_return_today.sum(axis=1),
    }).iloc[1:]  # the first session has no previous price, so no returns
    assert portfolio_daily_returns["snapshot_portfolio_return"].notna().all()
    assert (portfolio_daily_returns["weight_with_return"] <= 1 + 1e-12).all()
    assert (portfolio_daily_returns["snapshot_portfolio_return"] <= daily_simple_returns.iloc[1:].max(axis=1) + 1e-12).all()  # allow arithmetic rounding at the convex-combination boundary
    assert (portfolio_daily_returns["snapshot_portfolio_return"] >= daily_simple_returns.iloc[1:].min(axis=1) - 1e-12).all()
    return portfolio_daily_returns


**portfolio_exclusions**

One row per (date, ticker) with no return after the first session, with the reason it is missing.

Reasons, checked in this order: before_same_security_start, after_history_end, zero_volume_day, quarantined,
before_first_price, first_price_day, after_last_price, otherwise vendor_gap.


In [48]:
def portfolio_exclusions(daily_simple_returns, price_panels, snapshot_weights):
    """One row per (date, ticker) with no return after the first session, with the reason it is missing.

    Reasons, checked in this order: before_same_security_start, after_history_end, zero_volume_day, quarantined,
    before_first_price, first_price_day, after_last_price, otherwise vendor_gap.
    """
    adjusted_close = price_panels.adjusted_close
    first_usable_price_date = adjusted_close.apply(lambda prices: prices.first_valid_index())
    last_usable_price_date = adjusted_close.apply(lambda prices: prices.last_valid_index())
    zero_volume_today_or_yesterday = price_panels.zero_volume_days | price_panels.zero_volume_days.shift(1, fill_value=False)  # a return needs two prices

    missing_return_grid = daily_simple_returns.iloc[1:].isna()  # the first session has no returns for anyone and is not in the portfolio table
    missing_return_pairs = missing_return_grid.stack()
    missing_return_pairs = missing_return_pairs[missing_return_pairs].index  # (date, ticker) pairs with no return

    exclusion_dates = missing_return_pairs.get_level_values("date")
    exclusion_tickers = missing_return_pairs.get_level_values(1)
    first_date_for_row = first_usable_price_date.reindex(exclusion_tickers).to_numpy()
    last_date_for_row = last_usable_price_date.reindex(exclusion_tickers).to_numpy()
    before_start_for_row = price_panels.before_same_security_start.stack().reindex(missing_return_pairs).to_numpy()
    after_end_for_row = price_panels.after_history_end.stack().reindex(missing_return_pairs).to_numpy()
    zero_volume_for_row = zero_volume_today_or_yesterday.stack().reindex(missing_return_pairs).to_numpy()
    quarantined_for_row = price_panels.quarantined_returns.stack().reindex(missing_return_pairs).to_numpy()

    exclusion_reason = np.select(
        [before_start_for_row,                         # before the same security existed (predecessor or pre-listing data excluded)
         after_end_for_row,                            # after the end of the recorded history window
         zero_volume_for_row,                          # a price was masked by the zero-volume rule
         quarantined_for_row,                          # the extreme-event ledger quarantined this return or a price it needs
         exclusion_dates < first_date_for_row,         # before the first usable price
         exclusion_dates == first_date_for_row,        # first usable price, no prior price to compare with
         exclusion_dates > last_date_for_row],         # after the last usable price
        ["before_same_security_start", "after_history_end", "zero_volume_day", "quarantined", "before_first_price", "first_price_day", "after_last_price"],
        default="vendor_gap",                          # inside the trading span but a price is missing in the vendor data
    )
    return pd.DataFrame({
        "date": exclusion_dates,
        "ticker": exclusion_tickers,
        "snapshot_weight": snapshot_weights.reindex(exclusion_tickers).to_numpy(),
        "reason": exclusion_reason,
    })


**ticker_data_flags**

Per-ticker data flags (notebook 02's table) plus market_cap_nonpositive_days and quarantined_return_days.


In [49]:
def ticker_data_flags(daily_simple_returns, price_panels, snapshot_weights, rules, exclusions,
                      market_cap, market_cap_rejected, market_cap_missing_while_priced, last_session):
    """Per-ticker data flags (notebook 02's table) plus market_cap_nonpositive_days and quarantined_return_days."""
    adjusted_close = price_panels.adjusted_close
    returns_after_first_session = daily_simple_returns.iloc[1:]
    flags = pd.DataFrame({
        "snapshot_weight": snapshot_weights,
        "price_source": rules["price_source"].reindex(snapshot_weights.index).fillna("yahoo"),
        "same_security_history_start": pd.to_datetime(rules["history_start"]).reindex(snapshot_weights.index),
        "history_end": pd.to_datetime(rules["history_end"]).reindex(snapshot_weights.index),
        "first_usable_price_date": adjusted_close.apply(lambda prices: prices.first_valid_index()),
        "last_usable_price_date": adjusted_close.apply(lambda prices: prices.last_valid_index()),
        "sessions_with_return": returns_after_first_session.notna().sum(),
        "days_before_same_security_start": price_panels.before_same_security_start.sum(),
        "zero_volume_days_masked": price_panels.zero_volume_days.sum(),
        "vendor_gap_days": exclusions[exclusions["reason"] == "vendor_gap"].groupby("ticker").size().reindex(snapshot_weights.index, fill_value=0),
        "large_daily_moves": (returns_after_first_session.abs() > large_move_threshold).sum(),
        "market_cap_days_rejected": market_cap_rejected.sum(),
        "market_cap_days_missing_while_priced": market_cap_missing_while_priced.sum(),
    }).rename_axis("ticker")
    flags["under_one_year_history"] = flags["sessions_with_return"] < minimum_sessions_for_one_year
    flags["ends_before_last_session"] = flags["last_usable_price_date"] < pd.Timestamp(last_session)
    flags["size_dependent_exclusion_days"] = flags["market_cap_days_missing_while_priced"]  # formations on these days drop the name from size-dependent analysis only
    market_cap_nonpositive_days = (market_cap <= 0).sum().reindex(snapshot_weights.index, fill_value=0)  # kept in the descriptive table, missing for size (HONA 2026-06-26)
    quarantined_return_days = price_panels.quarantined_returns.iloc[1:].sum().reindex(snapshot_weights.index, fill_value=0)
    flags["any_flag"] = (
        flags.index.isin(rules.index)
        | (flags["zero_volume_days_masked"] > 0) | (flags["vendor_gap_days"] > 0) | (flags["large_daily_moves"] > 0)
        | (flags["market_cap_days_rejected"] > 0) | (flags["market_cap_days_missing_while_priced"] > 0)
        | flags["under_one_year_history"] | flags["ends_before_last_session"]
        | (market_cap_nonpositive_days > 0) | (quarantined_return_days > 0)
    )
    flags["market_cap_nonpositive_days"] = market_cap_nonpositive_days  # added by the foundation, after the original columns
    flags["quarantined_return_days"] = quarantined_return_days
    return flags


# ---------- table writers ----------


**build_same_security_rules_table**

Write same_security_rules_2026-09-30.csv from same_security.py and return the rules.


In [50]:
def build_same_security_rules_table(output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write same_security_rules_2026-09-30.csv from same_security.py and return the rules."""
    rules = same_security.same_security_rules()
    same_security_rules_path = Path(output_folder) / f"same_security_rules_{settings.holdings_date}.csv"
    rules.to_csv(same_security_rules_path)
    write_manifest(same_security_rules_path, [], built_by, source_modules=[same_security],
                   note="Same-security decisions (Sai, 13 Oct 2026): history window, price source and market-cap validity per exception, with the evidence.")
    return rules


**build_snapshot_portfolio_daily_returns**

Write the constant-weight snapshot portfolio's daily returns (renormalized over names with a return).


In [51]:
def build_snapshot_portfolio_daily_returns(through=settings.price_vintage_last_session, apply_corrections=True,
                                           output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write the constant-weight snapshot portfolio's daily returns (renormalized over names with a return)."""
    daily_simple_returns = loader.load_daily_returns(through=through, apply_corrections=apply_corrections)
    snapshot_weights = loader.load_snapshot_weights()
    portfolio_daily_returns = snapshot_portfolio_daily_returns(daily_simple_returns, snapshot_weights)
    table_path = _vintage_table_path(output_folder, "snapshot_portfolio_daily_returns", through)
    portfolio_daily_returns.to_csv(table_path)
    write_manifest(
        table_path,
        input_paths=[loader.universe_tickers_path(), loader.universe_weights_path(), loader.adjusted_close_path(), loader.split_adjusted_close_path(), loader.volume_path(), loader.supplement_path(),
                     Path(output_folder) / f"same_security_rules_{settings.holdings_date}.csv"] + _ledger_inputs(apply_corrections),
        built_by=built_by, source_modules=_source_modules(),
        note="Daily simple total return of the 30 Sep 2026 IVV snapshot held at constant weights. Each day the weights of names with a return "
             "are rescaled to sum to 1; weight_with_return records the share of snapshot weight covered. Zero-volume days are missing.",
        extra_fields={"corrections_applied": bool(_ledger_inputs(apply_corrections))},
    )
    return portfolio_daily_returns


**_build_etf_daily_returns**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [52]:
def _build_etf_daily_returns(load_function, prices_path, stem, return_name, note, through, output_folder, built_by):
    etf_daily_returns = load_function(through=through).iloc[1:].to_frame()  # the first session has no previous price
    assert etf_daily_returns[return_name].notna().all()
    table_path = _vintage_table_path(output_folder, stem, through)
    etf_daily_returns.to_csv(table_path)
    write_manifest(table_path, input_paths=[prices_path, loader.adjusted_close_path(), loader.split_adjusted_close_path(), loader.volume_path(), loader.supplement_path(), loader.universe_tickers_path()], built_by=built_by, source_modules=_source_modules(), note=note)
    return etf_daily_returns


**build_spy_daily_returns**

Write SPY daily simple total returns over the sessions of the stock pull.


In [53]:
def build_spy_daily_returns(through=settings.price_vintage_last_session, output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write SPY daily simple total returns over the sessions of the stock pull."""
    return _build_etf_daily_returns(loader.load_spy_returns, loader.spy_prices_path(), "spy_daily_returns", "spy_return",
                                    "Daily simple total return of SPY from Yahoo's adjusted close, over the sessions of the stock pull.",
                                    through, output_folder, built_by)


**build_smh_daily_returns**

Write SMH daily simple total returns over the sessions of the stock pull.


In [54]:
def build_smh_daily_returns(through=settings.price_vintage_last_session, output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write SMH daily simple total returns over the sessions of the stock pull."""
    return _build_etf_daily_returns(loader.load_smh_returns, loader.smh_prices_path(), "smh_daily_returns", "smh_return",
                                    "Daily simple total return of SMH from Yahoo's adjusted close, over the sessions of the stock pull.",
                                    through, output_folder, built_by)


**build_market_cap_tables**

Write ticker-level same-security market cap and issuer market cap (licensed; CSVs git-ignored).

Nonpositive values are kept in these descriptive tables and counted in the flags table; the study loader sets them
missing for size. Returns (market_cap, issuer_market_cap, multi_class_issuers).


In [55]:
def build_market_cap_tables(through=settings.price_vintage_last_session, apply_corrections=True,
                            output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write ticker-level same-security market cap and issuer market cap (licensed; CSVs git-ignored).

    Nonpositive values are kept in these descriptive tables and counted in the flags table; the study loader sets them
    missing for size. Returns (market_cap, issuer_market_cap, multi_class_issuers).
    """
    price_panels = loader.load_price_panels(through=through, apply_corrections=apply_corrections)
    market_cap = loader.load_market_cap_same_security(through=through, apply_corrections=apply_corrections, nonpositive_as_missing=False)
    identifiers = loader.load_identifiers()
    issuer_market_cap = loader.issuer_market_cap_from_tickers(market_cap, identifiers["cik"], identifiers["snapshot_weight"])
    assert market_cap.index.equals(price_panels.adjusted_close.index)

    multi_class_rows = []
    for cik, tickers_of_issuer in identifiers["cik"].groupby(identifiers["cik"]):
        class_tickers = list(tickers_of_issuer.index)
        if len(class_tickers) > 1:
            both_classes = market_cap[class_tickers].dropna()
            multi_class_rows.append({"cik": cik, "classes": ", ".join(class_tickers),
                                     "issuer_figure_from": identifiers.loc[class_tickers, "snapshot_weight"].idxmax(),
                                     "share_of_days_classes_identical": ((both_classes.max(axis=1) / both_classes.min(axis=1) - 1).abs() < 1e-9).mean()})
    multi_class_issuers = pd.DataFrame(multi_class_rows).set_index("cik")

    market_cap_same_security_path = _vintage_table_path(output_folder, "market_cap_same_security", through)
    issuer_market_cap_path = _vintage_table_path(output_folder, "issuer_market_cap", through)
    market_cap.to_csv(market_cap_same_security_path)
    issuer_market_cap.to_csv(issuer_market_cap_path)
    market_cap_inputs = ([loader.universe_tickers_path(), loader.market_cap_raw_path(), loader.adjusted_close_path(), loader.split_adjusted_close_path(), loader.volume_path(), loader.supplement_path(),
                          loader.ivv_equity_holdings_path(), loader.universe_weights_path(),
                          Path(output_folder) / f"same_security_rules_{settings.holdings_date}.csv"] + _ledger_inputs(apply_corrections))
    write_manifest(market_cap_same_security_path, market_cap_inputs, built_by, source_modules=_source_modules(),
        note="FMP market cap by ticker, kept only on days the same security has a usable price; WBD before 11 Apr 2022 rejected (wrong share basis). "
             "Derived from licensed vendor data: the CSV is git-ignored; this manifest is committed.",
        extra_fields={"nonpositive_values_kept": "Zero or negative values are kept here (HONA 2026-06-26) and counted in the flags table; "
                                                 "the study loader sets them missing for size measurement."})
    write_manifest(issuer_market_cap_path, market_cap_inputs, built_by, source_modules=_source_modules(),
        note="One market cap per issuer (columns are SEC CIKs). Multi-class issuers use the class with the larger snapshot weight "
             "(GOOGL, FOXA, NWSA); FMP prices the issuer's full share count at each class's own price, so classes are never summed. "
             "Derived from licensed vendor data: the CSV is git-ignored; this manifest is committed.",
        extra_fields={"nonpositive_values_kept": "Zero or negative values are kept here and set missing by the study loader for size measurement."})
    return market_cap, issuer_market_cap, multi_class_issuers


**build_portfolio_exclusions_and_flags**

Write the per-day portfolio exclusion log and the per-ticker flag table; returns (exclusions, flags).


In [56]:
def build_portfolio_exclusions_and_flags(through=settings.price_vintage_last_session, apply_corrections=True,
                                         output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Write the per-day portfolio exclusion log and the per-ticker flag table; returns (exclusions, flags)."""
    price_panels = loader.load_price_panels(through=through, apply_corrections=apply_corrections)
    daily_simple_returns = loader.daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns)
    snapshot_weights = loader.load_snapshot_weights()
    rules = same_security.same_security_rules()
    exclusions = portfolio_exclusions(daily_simple_returns, price_panels, snapshot_weights)

    portfolio_daily_returns = snapshot_portfolio_daily_returns(daily_simple_returns, snapshot_weights)
    excluded_weight_by_date = exclusions.groupby("date")["snapshot_weight"].sum().reindex(portfolio_daily_returns.index, fill_value=0)
    assert ((excluded_weight_by_date + portfolio_daily_returns["weight_with_return"] - 1).abs() < 1e-12).all()  # included plus excluded weight is the whole snapshot, every day

    market_cap, market_cap_rejected, market_cap_missing_while_priced = loader.clean_market_cap(
        loader._market_cap_raw(), loader.load_price_panels(through=settings.price_vintage_last_session, apply_corrections=apply_corrections).adjusted_close, rules)
    within_cutoff = market_cap.index <= pd.Timestamp(through)
    flags = ticker_data_flags(daily_simple_returns, price_panels, snapshot_weights, rules, exclusions,
                              market_cap[within_cutoff], market_cap_rejected[within_cutoff], market_cap_missing_while_priced[within_cutoff], through)

    portfolio_exclusions_path = _vintage_table_path(output_folder, "portfolio_exclusions", through)
    ticker_data_flags_path = _vintage_table_path(output_folder, "ticker_data_flags", through)
    exclusions.to_csv(portfolio_exclusions_path, index=False)
    flags.to_csv(ticker_data_flags_path)
    flag_inputs = ([loader.universe_tickers_path(), loader.universe_weights_path(), loader.adjusted_close_path(), loader.split_adjusted_close_path(), loader.volume_path(), loader.supplement_path(), loader.market_cap_raw_path(),
                    Path(output_folder) / f"same_security_rules_{settings.holdings_date}.csv"] + _ledger_inputs(apply_corrections))
    write_manifest(portfolio_exclusions_path, flag_inputs, built_by, source_modules=_source_modules(),
        note="One row per date and ticker left out of the snapshot portfolio renormalization. Reasons, checked in this order: before_same_security_start, "
             "after_history_end, zero_volume_day, quarantined, before_first_price, first_price_day, after_last_price, otherwise vendor_gap.")
    write_manifest(ticker_data_flags_path, flag_inputs, built_by, source_modules=_source_modules(),
        note=f"Per-ticker data flags. under_one_year_history means fewer than {minimum_sessions_for_one_year} returns; large_daily_moves counts daily "
             f"simple returns beyond +-{large_move_threshold:.0%}, flagged for review, not removed. size_dependent_exclusion_days counts priced days "
             "with no usable market cap. market_cap_nonpositive_days counts days with a zero or negative market cap (missing for size measurement, "
             "return kept); quarantined_return_days counts returns made missing by the extreme-event ledger.")
    return exclusions, flags


**build_all_processed_tables**

Regenerate every table notebook 02 writes, in dependency order; returns the written paths.


In [57]:
def build_all_processed_tables(through=settings.price_vintage_last_session, apply_corrections=True,
                               output_folder=settings.processed_data_folder, built_by=default_built_by):
    """Regenerate every table notebook 02 writes, in dependency order; returns the written paths."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    build_same_security_rules_table(output_folder, built_by)
    build_snapshot_portfolio_daily_returns(through, apply_corrections, output_folder, built_by)
    build_spy_daily_returns(through, output_folder, built_by)
    build_smh_daily_returns(through, output_folder, built_by)
    build_market_cap_tables(through, apply_corrections, output_folder, built_by)
    build_portfolio_exclusions_and_flags(through, apply_corrections, output_folder, built_by)
    return sorted(output_folder.glob("*.csv"))


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from quiet_index_500 import foundation, settings

universe_tickers = load_universe_tickers()  # the 503 snapshot tickers in frozen order, fingerprint-checked against notebook 01's manifest

universe_tickers[:10]

('NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'AVGO', 'GOOG', 'META', 'MU', 'TSLA')

## Collection boundary

This guarded cell is archival collection code. Full reproduction uses the exact saved vintage and does not refresh it.


In [2]:
import yfinance as yf

price_history_start = settings.price_history_start                  # start of year 2015; the first session is 2 Jan 2015
price_vintage_last_session = settings.price_vintage_last_session    # the frozen pull this notebook works from, named explicitly

repo_root_folder = settings.repository_root_folder
raw_data_folder = settings.raw_data_folder
adjusted_close_path = foundation.loader.adjusted_close_path()
split_adjusted_close_path = foundation.loader.split_adjusted_close_path()
volume_path = foundation.loader.volume_path()
price_pull_manifest_path = foundation.loader.price_pull_manifest_path()

if not price_pull_manifest_path.exists():  # download once; later runs read the saved files, so numbers do not shift with Yahoo's revisions
    new_york_now = pd.Timestamp.now(tz="America/New_York")
    todays_session_is_complete = new_york_now.hour >= 17  # an hour after the 16:00 close
    download_end_exclusive = (new_york_now.normalize() + pd.Timedelta(days=1 if todays_session_is_complete else 0)).strftime("%Y-%m-%d")  # yahoo's end date is exclusive

    yahoo_download = yf.download(
        list(universe_tickers),
        start=price_history_start,
        end=download_end_exclusive,
        auto_adjust=False,  # keep both: Adj Close (splits and dividends) and Close (splits only)
        actions=False,
        group_by="column",  # columns come back as (field, ticker)
        threads=True,
        progress=False,
    )

    adjusted_close_downloaded = yahoo_download["Adj Close"].reindex(columns=list(universe_tickers)).rename_axis("date")
    split_adjusted_close_downloaded = yahoo_download["Close"].reindex(columns=list(universe_tickers)).rename_axis("date")
    volume_downloaded = yahoo_download["Volume"].reindex(columns=list(universe_tickers)).rename_axis("date")

    sessions_with_prices = adjusted_close_downloaded.notna().any(axis=1)  # yahoo can post a session's volume before its closing prices; such a row is not a completed session
    adjusted_close_downloaded = adjusted_close_downloaded[sessions_with_prices]
    split_adjusted_close_downloaded = split_adjusted_close_downloaded[sessions_with_prices]
    volume_downloaded = volume_downloaded[sessions_with_prices]

    last_session_downloaded = adjusted_close_downloaded.index.max().strftime("%Y-%m-%d")
    assert last_session_downloaded == price_vintage_last_session, f"data ends {last_session_downloaded}; set price_vintage_last_session to that date and rerun"

    adjusted_close_downloaded.to_csv(adjusted_close_path)
    split_adjusted_close_downloaded.to_csv(split_adjusted_close_path)
    volume_downloaded.to_csv(volume_path)

    price_pull_manifest = {
        "source": "Yahoo Finance via yfinance",
        "yfinance_version": yf.__version__,
        "download_arguments": {"start": price_history_start, "end_exclusive": download_end_exclusive, "auto_adjust": False, "actions": False},
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "universe_file": "data/processed/universe_tickers_2026-09-30.csv",
        "tickers_requested": len(universe_tickers),
        "tickers_without_data": [ticker for ticker in universe_tickers if adjusted_close_downloaded[ticker].isna().all()],
        "first_session": adjusted_close_downloaded.index.min().strftime("%Y-%m-%d"),
        "last_session": last_session_downloaded,
        "sessions": len(adjusted_close_downloaded),
        "files": [
            {"file": file_path.resolve().relative_to(repo_root_folder).as_posix(), "field": field, "meaning": meaning,
             "sha256": hashlib.sha256(file_path.read_bytes()).hexdigest()}
            for file_path, field, meaning in [
                (adjusted_close_path, "Adj Close", "adjusted for splits and dividends; total-return basis"),
                (split_adjusted_close_path, "Close", "adjusted for splits only; the price for market cap"),
                (volume_path, "Volume", "shares traded, as Yahoo reports them"),
            ]
        ],
        "note": "Raw vendor data, saved as downloaded. The CSVs are git-ignored; this manifest is committed. "
                "Zero-volume days are kept here and treated as missing when loaded.",
    }
    price_pull_manifest_path.write_text(json.dumps(price_pull_manifest, indent=2) + "\n")

json.loads(price_pull_manifest_path.read_text())

{'source': 'Yahoo Finance via yfinance',
 'yfinance_version': '1.7.0',
 'download_arguments': {'start': '2015-01-01',
  'end_exclusive': '2026-10-14',
  'auto_adjust': False,
  'actions': False},
 'pulled_at_utc': '2026-10-14T01:16:23+00:00',
 'universe_file': 'data/processed/universe_tickers_2026-09-30.csv',
 'tickers_requested': 503,
 'tickers_without_data': [],
 'first_session': '2015-01-02',
 'last_session': '2026-10-12',
 'sessions': 2961,
 'files': [{'file': 'data/raw/prices_adjusted_close_through_2026-10-12.csv',
   'field': 'Adj Close',
   'meaning': 'adjusted for splits and dividends; total-return basis',
   'sha256': 'b4c7bbf846bd8ef55e092cb3cb27ae8358de2dbab828b3f8130a155cae42cad2'},
  {'file': 'data/raw/prices_close_split_adjusted_through_2026-10-12.csv',
   'field': 'Close',
   'meaning': 'adjusted for splits only; the price for market cap',
   'sha256': 'a2461ad76aed266a4f6eb1a5e63e33270c7468adbe1135a35a78ecce75a58acf'},
  {'file': 'data/raw/prices_volume_through_2026-10-

In [3]:
import requests

same_security_rules = foundation.same_security_rules()  # Sai, 13 Oct 2026: one row per exception with its evidence; written down in foundation/same_security.py

# recovered price histories: WBD from FMP, PSKY from Yahoo under its successor symbol
supplement_path = raw_data_folder / f"prices_supplement_through_{price_vintage_last_session}.csv"
supplement_manifest_path = raw_data_folder / f"prices_supplement_through_{price_vintage_last_session}.manifest.json"

if not supplement_manifest_path.exists():  # download once, over the window of the main pull
    fmp_api_key = Path("../local/fmp_api_key.txt").read_text().strip()  # git-ignored; never printed or written to a manifest

    def fmp_daily_prices(endpoint, symbol):  # one FMP end-of-day endpoint for one symbol, as a date-indexed frame
        response = requests.get(f"https://financialmodelingprep.com/stable/{endpoint}", timeout=60, params={
            "symbol": symbol, "from": price_history_start, "to": price_vintage_last_session, "apikey": fmp_api_key})
        response.raise_for_status()
        rows = pd.DataFrame(response.json())
        return rows.assign(date=pd.to_datetime(rows["date"])).set_index("date").sort_index()

    wbd_dividend_adjusted = fmp_daily_prices("historical-price-eod/dividend-adjusted", "WBD")  # adjClose: splits and dividends
    wbd_as_traded = fmp_daily_prices("historical-price-eod/full", "WBD")                       # close and volume as traded; no WBD splits in the window
    wbd_rows = pd.DataFrame({"ticker": "WBD", "source": "fmp", "source_symbol": "WBD",
                             "adjusted_close": wbd_dividend_adjusted["adjClose"],
                             "close_split_adjusted": wbd_as_traded["close"],
                             "volume": wbd_as_traded["volume"]})

    skyd_download = yf.download("SKYD", start=price_history_start,
                                end=(pd.Timestamp(price_vintage_last_session) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                                auto_adjust=False, actions=False, group_by="column", progress=False)
    psky_rows = pd.DataFrame({"ticker": "PSKY", "source": "yahoo", "source_symbol": "SKYD",
                              "adjusted_close": skyd_download["Adj Close"]["SKYD"],
                              "close_split_adjusted": skyd_download["Close"]["SKYD"],
                              "volume": skyd_download["Volume"]["SKYD"]})

    supplement_downloaded = pd.concat([wbd_rows, psky_rows]).rename_axis("date").reset_index()
    supplement_downloaded.to_csv(supplement_path, index=False)
    supplement_manifest = {
        "source": "WBD: Financial Modeling Prep end-of-day (dividend-adjusted and as-traded). PSKY: Yahoo Finance via yfinance under SKYD.",
        "download_arguments": {"from": price_history_start, "to": price_vintage_last_session},
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "rows_by_ticker": supplement_downloaded.groupby("ticker").size().to_dict(),
        "file": supplement_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(supplement_path.read_bytes()).hexdigest(),
        "note": "Saved as downloaded, full window. The same-security history windows in notebooks/02_prices.ipynb decide which rows are used. "
                "Vendor data: the CSV is git-ignored; this manifest is committed.",
    }
    supplement_manifest_path.write_text(json.dumps(supplement_manifest, indent=2) + "\n")

same_security_rules.drop(columns="evidence")

,history_start,history_end,price_source,market_cap_valid_from
ticker,,,,
WBD,NaN,2026-09-30,fmp:WBD,2022-04-11
PSKY,2025-08-07,NaN,yahoo:SKYD,NaN
SW,2024-07-08,NaN,yahoo,NaN
AMCR,2019-06-11,NaN,yahoo,NaN
FERG,2021-03-08,NaN,yahoo,NaN
FOX,2019-03-19,NaN,yahoo,NaN
FOXA,2019-03-19,NaN,yahoo,NaN


In [4]:
price_panels = load_price_panels(through=price_vintage_last_session)  # supplement, same-security windows, zero-volume rule and any ledger corrections, whole vintage

zero_volume_days_by_ticker = price_panels.zero_volume_days.sum()
display(zero_volume_days_by_ticker[zero_volume_days_by_ticker > 0].sort_values(ascending=False))

same_security_rules.drop(columns="evidence").assign(
    first_usable_price=price_panels.adjusted_close[same_security_rules.index].apply(lambda prices: prices.first_valid_index()),
    last_usable_price=price_panels.adjusted_close[same_security_rules.index].apply(lambda prices: prices.last_valid_index()),
    usable_prices=price_panels.adjusted_close[same_security_rules.index].notna().sum(),
)

HWM     35
VRT     12
PFG      2
IBKR     2
AMD      1
WTW      1
GDDY     1
DOC      1
SBAC     1
CHD      1
HUBB     1
CNC      1
CCI      1
DXCM     1
P        1
VST      1
XEL      1
BKR      1
GEN      1
dtype: int64

,history_start,history_end,price_source,market_cap_valid_from,first_usable_price,last_usable_price,usable_prices
ticker,,,,,,,
WBD,NaN,2026-09-30,fmp:WBD,2022-04-11,2015-01-02,2026-09-30,2953
PSKY,2025-08-07,NaN,yahoo:SKYD,NaN,2025-08-07,2026-10-12,297
SW,2024-07-08,NaN,yahoo,NaN,2024-07-08,2026-10-12,569
AMCR,2019-06-11,NaN,yahoo,NaN,2019-06-11,2026-10-12,1845
FERG,2021-03-08,NaN,yahoo,NaN,2021-03-08,2026-10-12,1407
FOX,2019-03-19,NaN,yahoo,NaN,2019-03-19,2026-10-12,1903
FOXA,2019-03-19,NaN,yahoo,NaN,2019-03-19,2026-10-12,1903


In [5]:
daily_simple_returns = daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns)  # r_t = P_t / P_t-1 - 1; missing after any missing price, never bridged

assert daily_simple_returns.shape == price_panels.adjusted_close.shape

daily_simple_returns.tail()

,NVDA,AAPL,MSFT,AMZN,GOOGL,AVGO,GOOG,META,MU,TSLA,...,NCLH,AOS,DVA,HRL,WYNN,ERIE,PSKY,MGM,BF-B,NWS
date,,,,,,,,,,,,,,,,,,,,,
2026-10-06,0.001423,0.002223,0.007845,0.019451,0.003492,0.036689,0.002210,-0.004071,-0.017294,0.005149,...,0.043040,-0.001229,0.009400,-0.009960,-0.007127,0.007947,-0.025562,0.012264,0.003045,-0.015000
2026-10-07,-0.007398,0.009112,0.000869,0.014164,0.008111,0.001863,0.008068,-0.023779,0.040591,-0.007539,...,-0.029658,-0.027417,-0.020066,-0.016600,-0.021535,-0.024415,-0.067156,-0.017682,-0.015560,0.000000
2026-10-08,-0.029435,0.011139,-0.013497,-0.022545,-0.006305,-0.043478,-0.007226,-0.000582,-0.047941,-0.007438,...,0.029236,0.022046,0.001358,-0.006650,0.004268,0.040272,0.043869,0.000333,0.023130,0.012690
2026-10-09,-0.005207,-0.011104,0.023842,0.032945,0.009676,0.003887,0.008699,-0.003080,-0.006603,0.020533,...,0.005165,-0.003182,0.012597,-0.011843,-0.001859,-0.019864,-0.012931,-0.024658,-0.005652,0.002819
2026-10-12,-0.012910,0.012862,0.011363,-0.011051,0.003014,-0.018836,0.003191,0.033799,-0.038834,-0.012203,...,-0.014130,0.003902,0.014114,0.005732,-0.002528,0.013601,0.000000,0.001708,0.015915,0.011559


## descriptive portfolio

`foundation.build_snapshot_portfolio_daily_returns` renormalizes available snapshot weights every day. That makes it useful descriptively, but it is not the fixed-holdings evaluation in 09.


In [6]:
processed_data_folder = settings.processed_data_folder

build_same_security_rules_table()  # written first: the manifests below list it as an input
snapshot_portfolio_daily_returns = build_snapshot_portfolio_daily_returns(through=price_vintage_last_session)  # constant snapshot weights, renormalized each day over names with a return

snapshot_portfolio_daily_returns.describe()

,snapshot_portfolio_return,weight_with_return,names_with_return
count,2960.000000,2960.000000,2960.000000
mean,0.001082,0.976987,481.726351
std,0.013091,0.016646,14.645251
min,-0.131872,0.934060,453.000000
25%,-0.004405,0.960261,468.000000
50%,0.001415,0.981248,483.000000
75%,0.007575,0.990019,494.000000
max,0.112727,1.000000,503.000000


In [7]:
spy_raw_path = foundation.loader.spy_prices_path()
spy_pull_manifest_path = raw_data_folder / f"prices_spy_through_{price_vintage_last_session}.manifest.json"

if not spy_pull_manifest_path.exists():  # download once, over exactly the sessions of the stock pull
    spy_download_end_exclusive = (pd.Timestamp(price_vintage_last_session) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    spy_download = yf.download("SPY", start=price_history_start, end=spy_download_end_exclusive,
                               auto_adjust=False, actions=False, group_by="column", progress=False)
    spy_prices_downloaded = pd.DataFrame({
        "adjusted_close": spy_download["Adj Close"]["SPY"],      # splits and dividends
        "close_split_adjusted": spy_download["Close"]["SPY"],    # splits only
        "volume": spy_download["Volume"]["SPY"],
    }).rename_axis("date")
    spy_prices_downloaded.to_csv(spy_raw_path)
    spy_pull_manifest = {
        "source": "Yahoo Finance via yfinance",
        "yfinance_version": yf.__version__,
        "ticker": "SPY",
        "download_arguments": {"start": price_history_start, "end_exclusive": spy_download_end_exclusive, "auto_adjust": False, "actions": False},
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "first_session": spy_prices_downloaded.index.min().strftime("%Y-%m-%d"),
        "last_session": spy_prices_downloaded.index.max().strftime("%Y-%m-%d"),
        "sessions": len(spy_prices_downloaded),
        "file": spy_raw_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(spy_raw_path.read_bytes()).hexdigest(),
        "note": "Raw vendor data, saved as downloaded. The CSV is git-ignored; this manifest is committed.",
    }
    spy_pull_manifest_path.write_text(json.dumps(spy_pull_manifest, indent=2) + "\n")

spy_daily_returns = build_spy_daily_returns(through=price_vintage_last_session)  # same zero-volume rule as the stocks

snapshot_portfolio_daily_returns[["snapshot_portfolio_return"]].join(spy_daily_returns).corr()

,snapshot_portfolio_return,spy_return
snapshot_portfolio_return,1.000000,0.970125
spy_return,0.970125,1.000000


## capitalization and issuer identity

`foundation.build_market_cap_tables` uses one representative class figure per issuer. Summing issuer-level capitalization across share classes would double-count company size.


In [8]:
import time
import requests

fmp_market_cap_url = "https://financialmodelingprep.com/stable/historical-market-capitalization"

market_cap_path = raw_data_folder / f"market_cap_fmp_through_{price_vintage_last_session}.csv"
market_cap_manifest_path = raw_data_folder / f"market_cap_fmp_through_{price_vintage_last_session}.manifest.json"

if not market_cap_manifest_path.exists():  # download once, over the same window as the price pull
    fmp_api_key = Path("../local/fmp_api_key.txt").read_text().strip()  # git-ignored; never printed or written to a manifest
    market_cap_by_ticker = {}
    tickers_failed = {}
    for ticker in universe_tickers:
        response = requests.get(fmp_market_cap_url, timeout=60, params={
            "symbol": ticker, "from": price_history_start, "to": price_vintage_last_session, "apikey": fmp_api_key})
        time.sleep(0.25)  # stay well inside the plan's rate limit
        rows = response.json() if response.ok else []
        if not rows:
            tickers_failed[ticker] = response.status_code
            continue
        one_ticker = pd.DataFrame(rows)
        market_cap_by_ticker[ticker] = one_ticker.assign(date=pd.to_datetime(one_ticker["date"])).set_index("date")["marketCap"]

    market_cap_downloaded = (
        pd.DataFrame(market_cap_by_ticker)
        .reindex(columns=list(universe_tickers))  # same column order as the prices; tickers with no data are all-missing columns
        .sort_index()
        .rename_axis("date")
    )
    market_cap_downloaded.to_csv(market_cap_path)

    market_cap_manifest = {
        "source": "Financial Modeling Prep, historical-market-capitalization endpoint (paid plan)",
        "source_url": fmp_market_cap_url,
        "download_arguments": {"from": price_history_start, "to": price_vintage_last_session},
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "tickers_requested": len(universe_tickers),
        "tickers_failed": tickers_failed,
        "first_date": market_cap_downloaded.index.min().strftime("%Y-%m-%d"),
        "last_date": market_cap_downloaded.index.max().strftime("%Y-%m-%d"),
        "rows": len(market_cap_downloaded),
        "file": market_cap_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(market_cap_path.read_bytes()).hexdigest(),
        "note": "Total market cap in USD, shares outstanding times close, as FMP computes it; not float-adjusted. "
                "Share counts step at FMP's update dates. Multi-class issuers carry the issuer total on every class "
                "(GOOGL and GOOG are identical), so summing classes double-counts. Licensed vendor data: the CSV is git-ignored.",
    }
    market_cap_manifest_path.write_text(json.dumps(market_cap_manifest, indent=2) + "\n")

market_cap, issuer_market_cap, multi_class_issuers = build_market_cap_tables(through=price_vintage_last_session)  # licensed: the CSVs are git-ignored, the manifests committed

multi_class_issuers

,classes,issuer_figure_from,share_of_days_classes_identical
cik,,,
1564708,"NWSA, NWS",NWSA,0.001689
1652044,"GOOGL, GOOG",GOOGL,1.000000
1754301,"FOXA, FOX",FOXA,0.005255


In [9]:
portfolio_exclusions, ticker_data_flags = build_portfolio_exclusions_and_flags(through=price_vintage_last_session)

display(portfolio_exclusions["reason"].value_counts())

ticker_data_flags[ticker_data_flags["any_flag"]].sort_values("snapshot_weight", ascending=False)

reason
before_first_price            52981
before_same_security_start     9836
zero_volume_day                  97
first_price_day                  46
after_history_end                 8
vendor_gap                        2
Name: count, dtype: int64

,snapshot_weight,price_source,same_security_history_start,history_end,first_usable_price_date,last_usable_price_date,sessions_with_return,days_before_same_security_start,zero_volume_days_masked,vendor_gap_days,large_daily_moves,market_cap_days_rejected,market_cap_days_missing_while_priced,under_one_year_history,ends_before_last_session,size_dependent_exclusion_days,any_flag,market_cap_nonpositive_days,quarantined_return_days
ticker,,,,,,,,,,,,,,,,,,,
AMD,0.015148,yahoo,NaT,NaT,2015-01-05,2026-10-12,2959,0,1,0,1,1,0,False,False,0,True,0,0
VRT,0.001409,yahoo,NaT,NaT,2018-08-02,2026-10-12,2035,0,12,0,0,15,0,False,False,0,True,0,0
HWM,0.001368,yahoo,NaT,NaT,2016-11-08,2026-10-12,2461,0,35,0,0,497,0,False,False,0,True,0,0
HOOD,0.001349,yahoo,NaT,NaT,2021-07-29,2026-10-12,1306,0,0,0,1,0,0,False,False,0,True,0,0
BE,0.001237,yahoo,NaT,NaT,2018-07-25,2026-10-12,2064,0,0,0,1,0,0,False,False,0,True,0,0
WBD,0.001179,fmp:WBD,NaT,2026-09-30,2015-01-02,2026-09-30,2952,0,0,0,0,1834,1831,False,True,1831,True,0,0
MRNA,0.001084,yahoo,NaT,NaT,2018-12-07,2026-10-12,1970,0,0,0,1,0,0,False,False,0,True,0,0
TRGP,0.000879,yahoo,NaT,NaT,2015-01-02,2026-10-12,2960,0,0,0,1,0,0,False,False,0,True,0,0
BKR,0.000827,yahoo,NaT,NaT,2015-01-02,2026-10-12,2958,0,1,0,0,1,0,False,False,0,True,0,0


In [10]:
smh_raw_path = foundation.loader.smh_prices_path()
smh_pull_manifest_path = raw_data_folder / f"prices_smh_through_{price_vintage_last_session}.manifest.json"

if not smh_pull_manifest_path.exists():  # download once, over exactly the sessions of the stock pull
    smh_download_end_exclusive = (pd.Timestamp(price_vintage_last_session) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    smh_download = yf.download("SMH", start=price_history_start, end=smh_download_end_exclusive,
                               auto_adjust=False, actions=False, group_by="column", progress=False)
    smh_prices_downloaded = pd.DataFrame({
        "adjusted_close": smh_download["Adj Close"]["SMH"],      # splits and dividends
        "close_split_adjusted": smh_download["Close"]["SMH"],    # splits only
        "volume": smh_download["Volume"]["SMH"],
    }).rename_axis("date")
    smh_prices_downloaded.to_csv(smh_raw_path)
    smh_pull_manifest = {
        "source": "Yahoo Finance via yfinance",
        "yfinance_version": yf.__version__,
        "ticker": "SMH",
        "download_arguments": {"start": price_history_start, "end_exclusive": smh_download_end_exclusive, "auto_adjust": False, "actions": False},
        "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "first_session": smh_prices_downloaded.index.min().strftime("%Y-%m-%d"),
        "last_session": smh_prices_downloaded.index.max().strftime("%Y-%m-%d"),
        "sessions": len(smh_prices_downloaded),
        "file": smh_raw_path.resolve().relative_to(repo_root_folder).as_posix(),
        "sha256": hashlib.sha256(smh_raw_path.read_bytes()).hexdigest(),
        "note": "VanEck Semiconductor ETF. Raw vendor data, saved as downloaded. The CSV is git-ignored; this manifest is committed.",
    }
    smh_pull_manifest_path.write_text(json.dumps(smh_pull_manifest, indent=2) + "\n")

smh_daily_returns = build_smh_daily_returns(through=price_vintage_last_session)

spy_daily_returns.join(smh_daily_returns).join(snapshot_portfolio_daily_returns[["snapshot_portfolio_return"]]).corr()

,spy_return,smh_return,snapshot_portfolio_return
spy_return,1.000000,0.802412,0.970125
smh_return,0.802412,1.000000,0.871082
snapshot_portfolio_return,0.970125,0.871082,1.000000


In [11]:
foundation.inventory_leftover_files()  # the 2026-10-13 pull: outside the study, left in place, listed here and never read

,path,bytes,sha256
0,data/raw/prices_adjusted_close_through_2026-10...,26039314,7e4dc38838a508a18fbcc455c84e66d6e2d0e3a2fe536a...
1,data/raw/prices_close_split_adjusted_through_2...,25478794,d15f6fbc89ff1cd09f4eedcc2043b2ad177fe52d822dd7...
2,data/raw/prices_through_2026-10-13.manifest.json,1443,0fb459b7dc234d59b93f7a55f9b20a634b4931ef67aee7...
3,data/raw/prices_volume_through_2026-10-13.csv,11506798,9083bfef1c6d2edcc7c51fdff0b9069e7b56f4c4ff9868...


## Findings

The study loader exposes 2,953 price sessions through September 30. Missing prices are never bridged into a daily return. The descriptive snapshot series renormalizes available weights each day; Notebook 09 uses a different, formation-fixed convention.

## Analysis and takeaways

The October 12 vintage and September 30 research cutoff are separate dates. Descriptive available-weight portfolio returns may renormalize weights daily; they do not supply the fixed-holding outcome in 09. The price series is a cleaned vendor-adjusted total-return proxy, with corporate-action conventions documented in 03b.

## Bridge through the study

03 aligns standard factors to price sessions; 03b verifies the cleaned foundation. Downstream functions use the shared loader rather than parsing vendor files independently.
